# Prepare working files

Clean working space

In [ ]:
import os, shutil

base = "/content"
for item in os.listdir(base):
    if item == "drive":
        continue
    path = os.path.join(base, item)
    if os.path.isdir(path):
        shutil.rmtree(path, ignore_errors=True)
    else:
        os.remove(path)
print("Finished cleaning")

Finished cleaning


Mount your google drive

In [ ]:
# Mount google drive to obtain dataset in zip format
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Get train_yolo files

In [ ]:
import os
import shutil

src_dir = "/content/drive/MyDrive/train_yolo"
dest_dir = "/content"

if not os.path.exists(src_dir):
    raise FileNotFoundError(f"Not found: {src_dir}")

moved = 0
for name in os.listdir(src_dir):
    src_path = os.path.join(src_dir, name)
    dest_path = os.path.join(dest_dir, name)

    if os.path.exists(dest_path):
        base, ext = os.path.splitext(name)
        k = 1
        new_name = f"{base}_{k}{ext}"
        while os.path.exists(os.path.join(dest_dir, new_name)):
            k += 1
            new_name = f"{base}_{k}{ext}"
        dest_path = os.path.join(dest_dir, new_name)

    if os.path.isdir(src_path):
        shutil.copytree(src_path, dest_path)
    else:
        shutil.copy2(src_path, dest_path)
    moved += 1

print("Finished moving train python files")


Finished moving train python files


Unzip dataset

In [ ]:
# Unzip zip format dataset here
import zipfile
import os

zip_name = input("Input zip dataset like train_yolo.zip): ").strip()

drive_path = "/content/drive/MyDrive"
zip_path = os.path.join(drive_path, zip_name)

if not os.path.exists(zip_path):
    raise FileNotFoundError(f"{drive_path} cant found {zip_name}")

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(".")

print(f"Finished unzip")


Input zip dataset like train_yolo.zip): sheep2.zip
Finished unzip


Input dataset folder name to make it usable in colab

In [ ]:
import os

def update_yaml_path():
    current_dir = os.getcwd()
    print(f"Current directory: {current_dir}")

    folder_name = input("Enter the folder name: ").strip()
    folder_path = os.path.join(current_dir, folder_name)

    if not os.path.exists(folder_path):
        print(f"Folder {folder_name} does not exist!")
        return

    print(f"Searching for .yaml files in {folder_path}...")

    for root, _, files in os.walk(folder_path):
        for file in files:
            if file.endswith(".yaml"):
                yaml_path = os.path.join(root, file)
                print(f"Found file: {yaml_path}")

                new_lines = []
                with open(yaml_path, "r", encoding="utf-8") as f:
                    for line in f:
                        if line.strip().startswith("path:"):
                            new_lines.append(f"path: {folder_path}\n")
                        else:
                            new_lines.append(line)

                with open(yaml_path, "w", encoding="utf-8") as f:
                    f.writelines(new_lines)

                print(f"Updated path in {yaml_path}")

update_yaml_path()

Current directory: /content
Enter the folder name: sheep
Searching for .yaml files in /content/sheep...
Found file: /content/sheep/data.yaml
✅ Updated path in /content/sheep/data.yaml


# Train (change hyperparameter if need)

Using train_yolo.py to train after you have dataset and python files mounted.

In [ ]:
!pip install ultralytics
!python3 -u train_yolo.py \
  --data ./sheep/data.yaml \
  --model yolov8s.pt \
  --epochs 100 \
  --imgsz 896 \
  --batch 16

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 29.0 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Dataset detected B style, converting to A ...
==> Config (from API)
data.yaml : /content/sheep/data.yaml
model     : yolov8s.pt
epochs    : 1
batch     : 16
imgsz     : 896
device    : cuda
project   : /content/sheep/runs
name      : exp
resume    : False
Ultralytics 8.3.190 🚀 Python-3.12.11 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/sheep/